In [28]:
#imports
import numpy as np
from torchvision import datasets as tv_datasets
from torchvision import transforms
import torch
import torch.nn as nn
import gc
import torch.nn.functional as F

In [29]:
#model definition
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class LNET(nn.Module):
    def __init__(self):
        super(LNET, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5, stride=1, padding=2)
        self.tanh1 = nn.Tanh()
        self.pool1 = nn.AvgPool2d(kernel_size=2,  stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1)
        self.tanh2 = nn.Tanh()
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(16, 120, kernel_size=5, stride=1)
        self.tanh3 = nn.Tanh()
        self.fc1 = nn.Linear(120, 84)
        self.tanh4 = nn.Tanh()
        self.fc2 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool1(self.tanh1(self.conv1(x)))
        x = self.pool2(self.tanh2(self.conv2(x)))
        x = self.tanh3(self.conv3(x))
        x = x.view(-1, 120)  # Flatten the tensor
        x = self.tanh4(self.fc1(x))
        x = self.fc2(x)
        return x

In [30]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x)
])

mnist = tv_datasets.MNIST(root='data', train=True, transform=transform, download=True)
mnist_loader = DataLoader(mnist, batch_size=1000, shuffle=True)
print("Loaded MNIST")

cifar = tv_datasets.CIFAR10(root='data', train=True, transform=transform, download=True)
cifar_loader = DataLoader(cifar, batch_size=1000, shuffle=True)
print("Loaded CIFAR-10")

Loaded MNIST
Loaded CIFAR-10


In [31]:
def get_gradient_vector(model, dataloader, device):
    model.to(device)
    model.train()
    x, y = next(iter(dataloader)) #1 batch
    x, y = x.to(device), y.to(device)
    
    model.zero_grad()        
    out = model(x)          
    loss = F.cross_entropy(out, y)   
    loss.backward()                  

    grad_vector = torch.cat([
        p.grad.view(-1) for p in model.parameters() if p.grad is not None
    ])
    
    return grad_vector.detach().cpu().numpy()


In [32]:
def compute_L(a_prime, r_values):
    """
    Compute L(a') = max over j,t of [max(r_{j,t} - 1/(1-a'), (1-a') - r_{j,t})]
    
    Parameters:
    a_prime: float in (0, 1], the variable to optimize
    r_values: 2D array of shape (J, T) containing r_{j,t} values
    
    Returns:
    L(a_prime): float, the objective function value
    """
    if a_prime <= 0 or a_prime >= 1:
        return float('inf')  # Outside valid domain
    
    term1 = r_values - 1/(1 - a_prime)  # r_{j,t} - 1/(1-a')
    term2 = (1 - a_prime) - r_values    # (1-a') - r_{j,t}
    
    # Take max of the two terms for each (j,t)
    inner_max = np.maximum(term1, term2)
    
    # Take max over all (j,t)
    L_value = np.max(inner_max)
    
    return L_value

def find_optimal_a_prime(r_values, bounds=(1e-8, 1-1e-8)):
    """
    Find the optimal a' that minimizes L(a') using golden section search
    
    Parameters:
    r_values: 2D array of r_{j,t} values
    bounds: tuple (lower, upper) bounds for a'
    
    Returns:
    result: dict with optimal a' and minimum L value
    """
    
    def objective(a_prime):
        return compute_L(a_prime, r_values)
    
    # Golden section search
    result = golden(objective, brack=bounds, tol=1e-10)
    return {'x': result, 'fun': objective(result), 'success': True}

In [33]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cifar_model = LNET()
mnist_model = LNET()

cifar_gradients = get_gradient_vector(cifar_model, cifar_loader, device)
mnist_gradients = get_gradient_vector(mnist_model, mnist_loader, device)



ValueError: Expected input batch_size (4000) to match target batch_size (1000).